# Brasileirão: notebook de partida

## A pergunta

Para cada partida futura do Brasileirão, quais são as probabilidades de vitória do mandante, de empate e de vitória do visitante?

## Como prosseguir

1. Rode tudo de cima a baixo.
2. Submeta o arquivo gerado e confirme seu nome no placar.
3. A partir daí, o caminho é seu. Melhore o modelo do jeito que quiser.

O método é escolha de vocês, dentro das bibliotecas permitidas. A única regra de ouro: usar apenas informação anterior a cada partida.

O modelo deste notebook é fraco de propósito. Ele usa só 2 das 17 colunas e marca cerca de 1.049 no placar privado, que vale nota 1. É o ponto de partida, não o destino: tudo acima disso vem do seu trabalho.

Métrica: LogLoss multiclasse. Quanto menor, melhor.

In [ ]:
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss

train = pd.read_csv('/kaggle/input/campeonato-inteli-modulo-3-2026/train.csv')
test  = pd.read_csv('/kaggle/input/campeonato-inteli-modulo-3-2026/test.csv')
print(train.shape, test.shape)
train.head()

## 1. O alvo

A coluna `Res` tem três valores: `H` (vitória do mandante), `D` (empate) e `A` (vitória do visitante).

Importante: o `log_loss` do scikit-learn ordena as classes em ordem alfabética (`A`, `D`, `H`). Monte suas colunas sempre nessa ordem. Este é o erro mais comum da competição. Colunas na ordem errada dão um score errado, sem nenhuma mensagem de aviso.

In [ ]:
CLASSES = ['A', 'D', 'H']   # ordem alfabetica, sempre
train['Res'].value_counts(normalize=True).round(3)

## 2. O ponto de partida sem modelo

O palpite mais simples: prever em toda partida a frequência histórica de cada resultado. Ele está fixado no placar como benchmark ("palpite ingênuo") e marca 1.063 no placar privado, que é onde a nota é calculada. Qualquer modelo de verdade precisa ficar abaixo disso.

In [ ]:
prior = np.array([(train.Res == c).mean() for c in CLASSES])
print('frequencias A/D/H:', prior.round(3))

## 3. Validação local

Você tem 5 submissões por dia. O placar público usa só 517 partidas e tem ruído de mais ou menos 0.018. Não dá para ajustar o modelo olhando o placar. Valide aqui, no notebook.

Os dados são temporais. Valide treinando no passado e testando no futuro. Não use `shuffle=True` e não embaralhe as linhas.

A função abaixo faz isso. Use com qualquer conjunto de colunas e qualquer modelo do scikit-learn.

In [ ]:
COLUNAS = ['elo_home','elo_away','elo_diff','form_pts_home','form_pts_away',
           'form_gf_home','form_ga_home','form_gf_away','form_ga_away',
           'home_form_as_host','away_form_as_visitor','rest_days_home',
           'rest_days_away','season_ppg_home','season_ppg_away','round_n','h2h_home_pts']

def valida(modelo, colunas, corte=2019):
    """Treina ate a temporada `corte`, valida nas seguintes. Retorna o LogLoss."""
    tr = train[train.Season <= corte]
    va = train[train.Season >  corte]
    modelo.fit(tr[colunas], tr['Res'])
    ordem = [list(modelo[-1].classes_).index(c) for c in CLASSES]
    p = modelo.predict_proba(va[colunas])[:, ordem]
    return log_loss(va['Res'], p, labels=CLASSES)

exemplo = make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=2000))
print('modelo de exemplo (2 colunas):',
      round(valida(exemplo, ['form_pts_home', 'form_pts_away']), 4))

Agora é com você. Troque as colunas, troque o modelo, compare os números que a função `valida` devolve. Só suba para o placar o que já melhorou aqui.

## 4. Gerar a submissão

In [ ]:
modelo = make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=2000))
modelo.fit(train[['form_pts_home', 'form_pts_away']], train['Res'])

ordem = [list(modelo[-1].classes_).index(c) for c in CLASSES]
probs = modelo.predict_proba(test[['form_pts_home', 'form_pts_away']])[:, ordem]

sub = pd.DataFrame({'Id': test['Id']})
for i, c in enumerate(CLASSES):
    sub[c] = probs[:, i]

# checagens antes de submeter
assert list(sub.columns) == ['Id','A','D','H'], 'colunas erradas'
assert len(sub) == len(test), 'numero de linhas errado'
assert np.allclose(sub[CLASSES].sum(axis=1), 1), 'probabilidades nao somam 1'

sub.to_csv('submission.csv', index=False)
sub.head()

## 5. As colunas de odds do treino

`odd_casa`, `odd_empate` e `odd_visitante` são as cotações que casas de apostas ofereciam nessas partidas. Elas não existem no `test.csv`.

Rode a célula abaixo e anote o resultado.

In [ ]:
imp = 1 / train[['odd_casa', 'odd_empate', 'odd_visitante']]
print('soma das probabilidades implicitas:', imp.sum(axis=1).mean().round(4))

Uma distribuição de probabilidade soma 1. Essa soma dá mais que 1. Por quê?

Anote sua hipótese. Vamos discutir na última semana.

## 6. Caminhos possíveis

Nenhum destes é obrigatório. São direções que outras pessoas já exploraram em problemas parecidos:

- Testar outras colunas e combinações de colunas
- Testar outros algoritmos de classificação
- Criar colunas novas a partir das existentes
- Agrupar times ou partidas parecidas e usar o grupo como informação nova
- Modelar o número de gols de cada time em vez do resultado direto
- Verificar se as probabilidades do seu modelo batem com a realidade (calibração)
- Combinar as previsões de mais de um modelo

Uma pergunta para levar durante toda a competição: quando seu modelo diz 60%, esse resultado acontece mesmo 6 vezes em 10?